# Training Pipeline - Part 2: XGBoost Model Training

**Purpose:** Train supervised gradient boosting classifier (Layer 3)

**Inputs:** X_train_scaled.pkl, X_val_scaled.pkl, y_train.pkl, y_val.pkl, X_test_scaled.pkl, y_test.pkl

**Outputs:** xgboost_model.pkl, xgboost_metrics.json

## Section 1: Imports and Setup

In [4]:
import os
import time
import json
import joblib
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')

from xgboost import XGBClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix, classification_report
)

os.makedirs('../trained_models', exist_ok=True)

print("✅ Imports complete")

✅ Imports complete


## Section 2 — Load Preprocessed Data

In [8]:
print("\n📂 LOADING PREPROCESSED DATA")
print("=" * 60)

X_train = joblib.load('../datasets/X_train_scaled.pkl').values
X_val   = joblib.load('../datasets/X_val_scaled.pkl').values
X_test  = joblib.load('../datasets/X_test_scaled.pkl').values
y_train = joblib.load('../datasets/y_train.pkl').values
y_val   = joblib.load('../datasets/y_val.pkl').values
y_test  = joblib.load('../datasets/y_test.pkl').values

with open('../trained_models/reverse_map.json', 'r', encoding='utf-8') as f:
    reverse_map = {int(k): v for k, v in json.load(f).items()}

num_classes  = len(reverse_map)
target_names = [reverse_map[i] for i in sorted(reverse_map.keys())]

print(f"✅ X_train : {X_train.shape}")
print(f"✅ X_val   : {X_val.shape}")
print(f"✅ X_test  : {X_test.shape}")
print(f"\nClasses ({num_classes}):")
for idx, name in sorted(reverse_map.items()):
    train_count = (y_train == idx).sum()
    print(f"  {idx:>2} → {name:<15} ({train_count:,})")


📂 LOADING PREPROCESSED DATA
✅ X_train : (1932175, 18)
✅ X_val   : (241522, 18)
✅ X_test  : (241522, 18)

Classes (15):
   0 → BENIGN          (1,598,452)
   1 → Bot             (1,559)
   2 → DDoS            (102,100)
   3 → DoS GoldenEye   (8,225)
   4 → DoS Hulk        (133,552)
   5 → DoS Slowhttptest (4,142)
   6 → DoS slowloris   (3,901)
   7 → FTP-Patator     (3,413)
   8 → Heartbleed      (9)
   9 → Infiltration    (29)
  10 → PortScan        (72,594)
  11 → SSH-Patator     (2,521)
  12 → Web Attack � Brute Force (1,141)
  13 → Web Attack � Sql Injection (16)
  14 → Web Attack � XSS (521)


## Section 3 — Class Imbalance Analysis

In [9]:
print("\n⚖️  CLASS IMBALANCE ANALYSIS")
print("=" * 60)

unique, counts = np.unique(y_train, return_counts=True)
total = len(y_train)

print(f"\n{'Class':>4}  {'Name':<15}  {'%':>6}  {'Weight':>8}")
print("-" * 70)

class_weights = {}
for cls, count in zip(unique, counts):
    pct = count / total * 100
    weight = total / (num_classes * count)
    class_weights[int(cls)] = round(weight, 4)
    print(f"  {cls:>2}   {reverse_map[int(cls)]:<15}  {pct:>5.2f}%  {weight:>8.2f}")

print(f"\n  Most common     : {reverse_map[int(unique[counts.argmax()])]} ({counts.max():,})")
print(f"  Rarest          : {reverse_map[int(unique[counts.argmin()])]} ({counts.min():,})")
print(f"  Imbalance ratio : {counts.max() / counts.min():.0f}:1")
print("\n  ✅ scale_pos_weight not used for multiclass")
print("  ✅ sample_weight passed per-sample to XGBoost instead")

sample_weights = np.array([class_weights[int(y)] for y in y_train])
print(f"\n  Sample weight range: {sample_weights.min():.4f} – {sample_weights.max():.4f}")


⚖️  CLASS IMBALANCE ANALYSIS

Class  Name                  %    Weight
----------------------------------------------------------------------
   0   BENIGN           82.73%      0.08
   1   Bot               0.08%     82.62
   2   DDoS              5.28%      1.26
   3   DoS GoldenEye     0.43%     15.66
   4   DoS Hulk          6.91%      0.96
   5   DoS Slowhttptest   0.21%     31.10
   6   DoS slowloris     0.20%     33.02
   7   FTP-Patator       0.18%     37.74
   8   Heartbleed        0.00%  14312.41
   9   Infiltration      0.00%   4441.78
  10   PortScan          3.76%      1.77
  11   SSH-Patator       0.13%     51.10
  12   Web Attack � Brute Force   0.06%    112.89
  13   Web Attack � Sql Injection   0.00%   8050.73
  14   Web Attack � XSS   0.03%    247.24

  Most common     : BENIGN (1,598,452)
  Rarest          : Heartbleed (9)
  Imbalance ratio : 177606:1

  ✅ scale_pos_weight not used for multiclass
  ✅ sample_weight passed per-sample to XGBoost instead

  Sample weigh

## Section 4 — Train XGBoost

In [10]:
print("\n🚀 TRAINING XGBOOST")
print("=" * 60)

xgb_model = XGBClassifier(
    n_estimators=500,
    max_depth=8,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=5,
    gamma=0.1,
    reg_alpha=0.1,
    reg_lambda=1.0,
    objective='multi:softprob',
    num_class=num_classes,
    eval_metric='mlogloss',
    tree_method='hist',
    random_state=42,
    n_jobs=-1,
    verbosity=1,
)

print("\nConfiguration:")
print("  n_estimators  : 500")
print("  max_depth     : 8")
print("  learning_rate : 0.05")
print(f"  num_classes   : {num_classes}")
print("  sample_weight  : per-class inverse frequency")
print("  tree_method    : hist (fast on large data)")

start_time = time.time()

xgb_model.fit(
    X_train, y_train,
    sample_weight=sample_weights,
    eval_set=[(X_val, y_val)],
    verbose=50
)

training_time = time.time() - start_time
print(f"\n✅ Training complete in {training_time:.1f}s ({training_time/60:.1f} min)")


🚀 TRAINING XGBOOST

Configuration:
  n_estimators  : 500
  max_depth     : 8
  learning_rate : 0.05
  num_classes   : 15
  sample_weight  : per-class inverse frequency
  tree_method    : hist (fast on large data)
[0]	validation_0-mlogloss:2.38250
[50]	validation_0-mlogloss:0.23820
[100]	validation_0-mlogloss:0.09455
[150]	validation_0-mlogloss:0.07669
[200]	validation_0-mlogloss:0.07077
[250]	validation_0-mlogloss:0.06768
[300]	validation_0-mlogloss:0.06613
[350]	validation_0-mlogloss:0.06516
[400]	validation_0-mlogloss:0.06417
[450]	validation_0-mlogloss:0.06364
[499]	validation_0-mlogloss:0.06313

✅ Training complete in 15598.4s (260.0 min)


In [5]:
# Sanity checks
assert X_train.shape[0] == len(y_train)
assert X_val.shape[0] == len(y_val)
assert X_test.shape[0] == len(y_test)

# Check for NA/inf
print('NA in X_train:', X_train.isna().any().any())
print('NA in X_val:', X_val.isna().any().any())
print('NA in X_test:', X_test.isna().any().any())
print('Any infs in X_train:', np.isinf(X_train).any())

# Train with GPU if available, otherwise CPU. Ensure 'create_model' is defined earlier.
start = time.time()
try:
    xgb_model = create_model(use_gpu=True)
    xgb_model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        verbose=50,
        early_stopping_rounds=20
    )
except Exception as e:
    print('⚠️ GPU failed or not available, switching to CPU. Error:', e)
    xgb_model = create_model(use_gpu=False)
    xgb_model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        verbose=50,
        early_stopping_rounds=20
    )
training_time = time.time() - start

print(f"✅ Training complete in {training_time:.2f} seconds")

NameError: name 'X_train' is not defined

## Section 5 — Evaluate

In [ ]:
print("\n📊 XGBOOST EVALUATION")
print("=" * 60)

# Validation
y_val_pred = xgb_model.predict(X_val)
y_val_proba = xgb_model.predict_proba(X_val)

val_acc = accuracy_score(y_val, y_val_pred)
val_prec = precision_score(y_val, y_val_pred, average='weighted', zero_division=0)
val_rec = recall_score(y_val, y_val_pred, average='weighted', zero_division=0)
val_f1 = f1_score(y_val, y_val_pred, average='weighted', zero_division=0)

print("\n1. Validation Performance:")
print(f"   Accuracy  : {val_acc:.4f}")
print(f"   Precision : {val_prec:.4f}  (weighted)")
print(f"   Recall    : {val_rec:.4f}  (weighted)")
print(f"   F1-Score  : {val_f1:.4f}  (weighted)")

# Test
y_pred = xgb_model.predict(X_test)
y_pred_proba = xgb_model.predict_proba(X_test)

acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred, average='weighted', zero_division=0)
rec = recall_score(y_test, y_pred, average='weighted', zero_division=0)
f1 = f1_score(y_test, y_pred, average='weighted', zero_division=0)

print("\n2. Test Performance:")
print(f"   Accuracy  : {acc:.4f}")
print(f"   Precision : {prec:.4f}  (weighted)")
print(f"   Recall    : {rec:.4f}  (weighted)")
print(f"   F1-Score  : {f1:.4f}  (weighted)")

try:
    roc_auc = roc_auc_score(y_test, y_pred_proba, multi_class='ovr', average='weighted')
    print(f"   ROC-AUC   : {roc_auc:.4f}  (weighted OvR)")
except Exception as e:
    roc_auc = None
    print(f"   ROC-AUC   : N/A ({e})")

print("\n3. Classification Report (Test):")
print(classification_report(y_test, y_pred, target_names=target_names, zero_division=0))

print("4. Per-class breakdown:")
print(f"   {'Class':<15} {'Precision':>10} {'Recall':>8} {'F1':>8}")
print(f"   {'-'*48}")
for idx, name in sorted(reverse_map.items()):
    mask = y_test == idx
    support = mask.sum()
    if support == 0:
        continue
    p = precision_score(mask.astype(int), (y_pred == idx).astype(int), zero_division=0)
    r = recall_score(mask.astype(int), (y_pred == idx).astype(int), zero_division=0)
    f = f1_score(mask.astype(int), (y_pred == idx).astype(int), zero_division=0)
    flag = '  ⚠️  low recall' if r < 0.5 else ''
    print(f"  {name:<15} {p:>10.4f} {r:>8.4f} {f:>8.4f}{flag}")

In [42]:
print('📊 MODEL EVALUATION')
print('=' * 80)

# --- Validation ---
y_val_pred = xgb_model.predict(X_val)
y_val_proba = xgb_model.predict_proba(X_val)
val_accuracy = accuracy_score(y_val, y_val_pred)
val_precision = precision_score(y_val, y_val_pred, average='weighted', zero_division=0)
val_recall = recall_score(y_val, y_val_pred, average='weighted', zero_division=0)
val_f1 = f1_score(y_val, y_val_pred, average='weighted', zero_division=0)

print(f'\n1. Validation Performance:\n   Accuracy:  {val_accuracy:.4f}')
print(f'   Precision: {val_precision:.4f}')
print(f'   Recall:    {val_recall:.4f}')
print(f'   F1-Score:  {val_f1:.4f}')

# --- Test ---
y_pred = xgb_model.predict(X_test)
y_pred_proba = xgb_model.predict_proba(X_test)
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average='weighted', zero_division=0)
recall = recall_score(y_test, y_pred, average='weighted', zero_division=0)
f1 = f1_score(y_test, y_pred, average='weighted', zero_division=0)

print(f'\n2. Test Performance:\n   Accuracy:  {accuracy:.4f}')
print(f'   Precision: {precision:.4f}')
print(f'   Recall:    {recall:.4f}')
print(f'   F1-Score:  {f1:.4f}')

try:
    roc_auc = roc_auc_score(y_test, y_pred_proba, multi_class='ovr')
    print(f'   ROC-AUC:   {roc_auc:.4f}')
except ValueError:
    roc_auc = None
    print('   ROC-AUC:   N/A (some classes missing from predictions)')

# --- Per-class report ---
print('\n3. Per-Class Metrics:')
if label_encoder is not None and hasattr(label_encoder, 'classes_'):
    target_names = [str(c) for c in label_encoder.classes_]
else:
    target_names = [str(i) for i in sorted(np.unique(y_test))]
print(classification_report(y_test, y_pred, target_names=target_names, zero_division=0))

# --- Confusion matrix ---
cm = confusion_matrix(y_test, y_pred)
print(f'\n4. Confusion Matrix:\n{cm}')

📊 MODEL EVALUATION

1. Validation Performance:
   Accuracy:  0.9971
   Precision: 0.9970
   Recall:    0.9971
   F1-Score:  0.9967

2. Test Performance:
   Accuracy:  0.9974
   Precision: 0.9973
   Recall:    0.9974
   F1-Score:  0.9969
   ROC-AUC:   0.9920

3. Per-Class Metrics:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00    199807
           1       1.00      0.39      0.57       195
           2       1.00      1.00      1.00     12763
           3       0.96      0.99      0.97      1028
           4       1.00      0.99      1.00     16694
           5       0.99      0.99      0.99       518
           6       1.00      0.98      0.99       487
           7       1.00      1.00      1.00       426
           8       1.00      1.00      1.00         1
           9       1.00      0.50      0.67         4
          10       0.99      1.00      0.99      9075
          11       0.98      0.98      0.98       315
          12    

## Section 6 — Feature Importance

In [ ]:
print("\n🔍 FEATURE IMPORTANCE")
print("=" * 60)

with open('../trained_models/feature_names.json', 'r', encoding='utf-8') as f:
    feature_names = json.load(f)

importances = xgb_model.feature_importances_
importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': importances
}).sort_values('Importance', ascending=False).reset_index(drop=True)

print("\nTop 10 features:")
print(f"  {'Rank':<4} {'Feature':<35} {'Importance':>12}")
print(f"  {'-'*55}")
for i, row in importance_df.head(10).iterrows():
    print(f"  {i+1:<4} {row['Feature']:<35} {row['Importance']:>12.6f}")

## Section 7 — Plots

In [ ]:
print("\n📈 GENERATING PLOTS")
print("=" * 60)

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

im = axes[0].imshow(cm_norm, cmap='Blues', vmin=0, vmax=1)
axes[0].set_xticks(range(num_classes))
axes[0].set_yticks(range(num_classes))
short_names = [n[:12] for n in target_names]
axes[0].set_xticklabels(short_names, rotation=45, ha='right', fontsize=8)
axes[0].set_yticklabels(short_names, fontsize=8)
axes[0].set_title('XGBoost — Normalised Confusion Matrix (Test)')
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('True')
plt.colorbar(im, ax=axes[0])

for r in range(num_classes):
    for c in range(num_classes):
        val = cm_norm[r, c]
        if val > 0.01:
            axes[0].text(
                c, r, f'{val:.2f}',
                ha='center', va='center',
                fontsize=7,
                color='white' if val > 0.5 else 'black'
            )

# Feature importance
top15 = importance_df.head(15)
axes[1].barh(top15['Feature'][::-1], top15['Importance'][::-1], color='steelblue')
axes[1].set_title('XGBoost — Top 15 Feature Importances')
axes[1].set_xlabel('Importance Score')
axes[1].grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.savefig('../trained_models/xgboost_plots.png', dpi=150)
plt.show()
print('✅ Saved: xgboost_plots.png')

## Section 8 — Save Model and Metrics

In [ ]:
print("\n💾 SAVING XGBOOST MODEL AND METRICS")
print("=" * 60)

joblib.dump(xgb_model, '../trained_models/xgboost_model.pkl')
print('✅ xgboost_model.pkl')

label_encoder_path = '../trained_models/label_encoder.pkl'
if os.path.exists(label_encoder_path):
    print('✅ label_encoder.pkl  (from preprocessing)')
else:
    print('⚠️  label_encoder.pkl missing — check 01_preprocessing.ipynb')

metrics_out = {
    'model_type': 'XGBoost Gradient Boosting',
    'num_classes': int(num_classes),
    'feature_count': int(X_train.shape[1]),
    'n_estimators': int(xgb_model.n_estimators),
    'max_depth': int(xgb_model.max_depth),
    'training_time_sec': float(training_time),
    'train_samples': int(len(y_train)),
    'val_samples': int(len(y_val)),
    'test_samples': int(len(y_test)),
    'val_accuracy': float(val_acc),
    'val_precision': float(val_prec),
    'val_recall': float(val_rec),
    'val_f1_score': float(val_f1),
    'accuracy': float(acc),
    'precision': float(prec),
    'recall': float(rec),
    'f1_score': float(f1),
    'roc_auc': float(roc_auc) if roc_auc is not None else None,
    'class_mapping': {str(k): v for k, v in reverse_map.items()},
    'top_features': importance_df.head(10)[['Feature', 'Importance']].to_dict(orient='records')
}

with open('../trained_models/xgboost_metrics.json', 'w', encoding='utf-8') as f:
    json.dump(metrics_out, f, indent=2)
print('✅ xgboost_metrics.json')

importance_df.to_csv('../trained_models/xgboost_feature_importance.csv', index=False)
print('✅ xgboost_feature_importance.csv')

print(f"\n{'='*60}")
print('FINAL SUMMARY')
print(f"{'='*60}")
print(f"  Val  Accuracy : {val_acc:.4f}  |  F1 : {val_f1:.4f}")
print(f"  Test Accuracy : {acc:.4f}  |  F1 : {f1:.4f}")
print(f"  Training time : {training_time/60:.1f} min")
print(f"  Classes       : {num_classes}")
print(f"\n✅ XGBoost notebook complete!")
print(f"   Next → 03_bert.ipynb")